# Drought Detection — HuggingFace Transformer Approach

Builds on notebook 01. Progression:
- **Step 1**: Zero-shot (no training, instant results)
- **Step 2**: ClimateBERT embeddings + simple classifier
- **Step 3**: Full fine-tuning (most powerful)

Each step teaches a new concept. Run them in order.

## Step 1: Install HuggingFace Libraries

In [ ]:
!pip install transformers torch datasets scikit-learn pandas numpy tqdm

## Step 2: What Is a Transformer? (Concept)

**TF-IDF** (notebook 01): counts word frequencies. No understanding of meaning.

**Transformer / BERT**: reads whole sentence, understands context.
- `drought` near `crops` → different meaning than `drought` near `policy`
- Pre-trained on billions of sentences → already knows language
- We add a small classifier head on top → fine-tune for our task

**ClimateBERT** = BERT already fine-tuned on climate/environment text.
Starting from ClimateBERT means it already understands words like `arid`, `reservoir`, `precipitation`.

## Step 3: Zero-Shot Classification (No Training Needed)

**Concept**: BART model trained on Natural Language Inference (NLI).
NLI = does sentence A entail sentence B?

We ask: does this headline entail the label `this text is about drought`?
No training data needed at all — works out of the box.

In [ ]:
from transformers import pipeline

# Downloads ~1.6GB first time — be patient
zero_shot = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

headlines = [
    'Severe drought threatens crops across southern Australia',
    'Stock markets rally on strong earnings reports',
    'Water reservoirs hit record low as dry spell continues',
    'New smartphone model released with improved camera',
    'Farmers face ruin as drought decimates harvest',
    'California water restrictions tighten as reservoir levels drop',
]

candidate_labels = ['drought or water crisis', 'unrelated news']

print('Zero-shot results:\n')
for h in headlines:
    result = zero_shot(h, candidate_labels)
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    drought_flag = '✓ DROUGHT' if 'drought' in top_label else '  other'
    print(f'{drought_flag} ({top_score:.2f})  {h[:60]}')

## Step 4: Load ClimateBERT (Concept)

**Why ClimateBERT over plain BERT?**

Plain BERT trained on Wikipedia + books. ClimateBERT further trained on:
- IPCC reports
- Climate news articles
- Scientific abstracts

Domain vocabulary (`aquifer`, `desertification`, `hydrological`) already understood.
This gives better embeddings for our task with less data.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

MODEL_NAME = 'climatebert/distilroberta-base-climate-detector'

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()
print('Model loaded.')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Step 5: Extract Embeddings (Concept)

**Embedding** = dense vector representing meaning of text.

We pass each headline through ClimateBERT → get 768-dimensional vector.
Similar meaning → similar vectors (close in space).

We use `[CLS]` token embedding = summary of whole sentence.
Then train a simple Logistic Regression on these vectors.

In [ ]:
from tqdm import tqdm

SAMPLE_DATA = [
    {'text': 'Severe drought threatens crops across southern Australia', 'label': 1},
    {'text': 'California water restrictions tighten as reservoir levels drop', 'label': 1},
    {'text': 'Horn of Africa drought leaves millions facing food crisis', 'label': 1},
    {'text': 'Kenya declares national drought emergency', 'label': 1},
    {'text': 'Water scarcity forces farmers to abandon fields in Spain', 'label': 1},
    {'text': 'Amazon basin faces worst drought in recorded history', 'label': 1},
    {'text': 'Indian farmers protest as monsoon fails for second year', 'label': 1},
    {'text': 'Western US faces multi-year megadrought conditions', 'label': 1},
    {'text': 'Ethiopia facing worst water shortage in decades', 'label': 1},
    {'text': 'Murray-Darling basin drought devastates Australian agriculture', 'label': 1},
    {'text': 'Zimbabwe declares state of disaster over drought', 'label': 1},
    {'text': 'Southern Europe braces for another summer of extreme drought', 'label': 1},
    {'text': 'New smartphone model released with improved camera system', 'label': 0},
    {'text': 'Stock markets rally as earnings season beats expectations', 'label': 0},
    {'text': 'Local football team wins championship after dramatic final', 'label': 0},
    {'text': 'Film festival announces lineup for upcoming season', 'label': 0},
    {'text': 'Electric vehicle sales surge in European market', 'label': 0},
    {'text': 'Scientists discover new species of deep-sea fish', 'label': 0},
    {'text': 'Central bank raises interest rates to combat inflation', 'label': 0},
    {'text': 'Streaming platform announces major original series renewal', 'label': 0},
    {'text': 'New museum wing opens showcasing ancient civilisations', 'label': 0},
    {'text': 'City council approves new public transport expansion plan', 'label': 0},
    {'text': 'Tech startup raises record funding round for AI research', 'label': 0},
    {'text': 'Olympic committee announces host city for next games', 'label': 0},
]

import pandas as pd
df = pd.DataFrame(SAMPLE_DATA)

def get_embedding(text):
    """Pass text through ClimateBERT, return [CLS] embedding."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    # [CLS] token is first token of last hidden state
    cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
    return cls_embedding

print('Extracting embeddings...')
embeddings = np.array([get_embedding(t) for t in tqdm(df['text'])])
print(f'Embedding shape: {embeddings.shape}')  # (n_samples, 768)
print('Done.')

## Step 6: Train Classifier on Embeddings

Same Logistic Regression as notebook 01 — but inputs are now rich semantic vectors
from ClimateBERT instead of sparse TF-IDF counts.

Compare results to see the improvement.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report

X = embeddings
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('ClimateBERT + Logistic Regression:')
print(classification_report(y_test, y_pred, target_names=['Other', 'Drought']))

# Cross-validation (more reliable with small dataset)
cv_scores = cross_val_score(clf, X, y, cv=5, scoring='f1')
print(f'5-fold CV F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

## Step 7: Visualise Embedding Space

**Concept**: reduce 768 dimensions → 2D using t-SNE to visualise clustering.

If ClimateBERT embeddings are good, drought articles should cluster together,
other articles in separate cluster.

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

tsne = TSNE(n_components=2, random_state=42, perplexity=min(10, len(df)-1))
emb_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(8, 6))
colors = ['steelblue' if l == 0 else 'tomato' for l in y]
plt.scatter(emb_2d[:, 0], emb_2d[:, 1], c=colors, s=80, alpha=0.8)

from matplotlib.patches import Patch
plt.legend(handles=[Patch(color='tomato', label='Drought'), Patch(color='steelblue', label='Other')])
plt.title('ClimateBERT Embeddings — t-SNE 2D')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()
print('Tight clusters = model separates drought vs other well.')

## Step 8: Use ClimateBERT's Own Classifier Head (Concept)

The `distilroberta-base-climate-detector` model already has a classification head.
It classifies text as `climate` vs `not climate`.

We can use this directly — then filter climate articles further for drought specifically.

In [ ]:
from transformers import pipeline as hf_pipeline

# Uses the built-in classifier head from ClimateBERT
climate_detector = hf_pipeline(
    'text-classification',
    model='climatebert/distilroberta-base-climate-detector'
)

test_headlines = [
    'Severe drought threatens crops across southern Australia',
    'Stock markets rally on strong earnings reports',
    'Water reservoirs hit record low as dry spell continues',
    'New smartphone model released with improved camera',
    'Farmers face ruin as drought decimates harvest',
]

print('ClimateBERT climate detection:\n')
for h in test_headlines:
    result = climate_detector(h)[0]
    flag = '✓ CLIMATE' if result['label'] == 'yes' else '  not climate'
    print(f'{flag} ({result["score"]:.2f})  {h[:60]}')

print('\nNote: climate = yes is a prerequisite for drought detection.')
print('Next step: filter climate=yes articles, then run drought keyword/classifier.')

## Step 9: Two-Stage Pipeline (Concept + Code)

**Stage 1**: ClimateBERT → is this climate-related?
**Stage 2**: Drought classifier → is this specifically about drought?

This is more precise than one single classifier.

In [ ]:
DROUGHT_KEYWORDS = [
    'drought', 'droughts', 'water shortage', 'water scarcity', 'water crisis',
    'dry spell', 'water stress', 'rainfall deficit', 'groundwater depletion',
    'reservoir levels', 'water restrictions', 'soil moisture deficit',
    'hydrological drought', 'meteorological drought', 'agricultural drought',
    'parched', 'water deficit'
]

def two_stage_predict(headlines):
    results = []
    for h in headlines:
        # Stage 1: climate filter
        climate_result = climate_detector(h)[0]
        is_climate = climate_result['label'] == 'yes'
        climate_conf = climate_result['score']

        # Stage 2: drought check (keyword + embedding classifier)
        if is_climate:
            emb = get_embedding(h).reshape(1, -1)
            drought_prob = clf.predict_proba(emb)[0][1]
            keyword_hit = any(kw in h.lower() for kw in DROUGHT_KEYWORDS)
            is_drought = drought_prob > 0.5 or keyword_hit
        else:
            drought_prob = 0.0
            keyword_hit = False
            is_drought = False

        results.append({
            'headline': h[:60],
            'climate': is_climate,
            'drought': is_drought,
            'drought_prob': round(drought_prob, 3),
        })
    return pd.DataFrame(results)

test = [
    'Severe drought threatens crops across southern Australia',
    'Stock markets rally on strong earnings reports',
    'Water reservoirs hit record low as dry spell continues',
    'New smartphone model released with improved camera',
    'Farmers face ruin as drought decimates harvest',
    'Climate activists march in capital city',
    'Renewable energy investment hits record high',
]

two_stage_predict(test)

## Step 10: What Next? — Fine-Tuning

**Concept**: instead of frozen embeddings + classifier, update ALL weights on drought data.

Requires:
- ~500+ labeled examples (more = better)
- GPU recommended (CPU possible but slow)
- HuggingFace `Trainer` API

When you have real labeled data from Step 4B (RSS/NewsAPI),
the next notebook will cover full fine-tuning with `Trainer`.

**Current pipeline accuracy order** (low → high):
1. Keyword baseline (notebook 01)
2. TF-IDF + LR (notebook 01)
3. ClimateBERT embeddings + LR (this notebook, Step 6)
4. Two-stage pipeline (this notebook, Step 9)
5. Full fine-tune ClimateBERT (next notebook)